# Optimized Gradient Method without $z_k$

This notebook applies the systematic Lyapunov-function discovery procedure to the OGM1 formulation 
of the Optimized Gradient Method, without the explicit $z_k$ iterates. The point of this version is
 that we the auxiliary vector $z_k$ is deliberately hidden, and we recover this missing direction 
 by the basis completion procedure implemented by `complete_basis_with_sparsifying_last_vector`.

## Import the required libraries

In [1]:
import functools
import itertools

import numpy as np
import pepflow as pf
import pepflow.lyapunov_utils as lu
import sympy as sp
from IPython.display import display
import IPython.display as ipd

## Define the function class

In [2]:
L = pf.Parameter("L")
f = pf.SmoothConvexFunction(is_basis=True, tags=["f"], L=L)

## Write a function that generates a PEPContext object associated with OGM1

In [3]:
@functools.cache
def theta(i: int, N: int):
    if i == -1:
        return 0
    if i == N:
        return 1 / 2 * (1 + np.sqrt(8 * theta(N - 1, N) ** 2 + 1))
    return 1 / 2 * (1 + np.sqrt(4 * theta(i - 1, N) ** 2 + 1))


def make_ctx_ogm1(
    ctx_name: str, N: int | sp.Integer, stepsize: pf.Parameter
) -> pf.PEPContext:
    N = int(N)
    ctx_ogm = pf.PEPContext(ctx_name).set_as_current()
    x = pf.Vector(is_basis=True, tags=["x_0"])
    f.set_stationary_point("x_star")

    for i in range(N):
        y = x - stepsize * f.grad(x)
        y.add_tag(f"y_{i + 1}")
        y_prev = ctx_ogm["x_0"] if i == 0 else ctx_ogm[f"y_{i}"]
        x = (
            y
            + (theta(i, N) - 1) / theta(i + 1, N) * (y - y_prev)
            + theta(i, N) / theta(i + 1, N) * (y - x)
        )
        x.add_tag(f"x_{i + 1}")

    return ctx_ogm

## Numerical PEP solve
### Find tight and sparsified proof certificates:

In [4]:
N = 5
N_int = int(N)
R = pf.Parameter("R")
L_value = sp.S(1)  # erase
R_value = sp.S(1)

ctx_prf = make_ctx_ogm1(ctx_name="ctx_prf", N=N, stepsize=1 / L)
pb_prf = pf.PEPBuilder(ctx_prf)
pb_prf.add_initial_constraint(
    ((ctx_prf["x_0"] - ctx_prf["x_star"]) ** 2).le(R, name="initial_condition")
)
pb_prf.set_performance_metric(f(ctx_prf[f"x_{N_int}"]) - f(ctx_prf["x_star"]))

result_dense = pb_prf.solve(resolve_parameters={"L": L_value, "R": R_value})
lamb_dense = result_dense.get_scalar_constraint_dual_value_in_numpy(f)

print("dense optimum:", result_dense.opt_value)

dense optimum: 0.01858694825943362


In [5]:
pf.pprint_labeled_matrix(lamb_dense.matrix, lamb_dense.row_names, lamb_dense.col_names)

<IPython.core.display.Math object>

### The $\lambda$ certificates have the same sparse pattern as the OGM2 formulation, so we enforce the observed relaxation and solve again

In [6]:
def tag_to_index(tag: str, N: int | sp.Integer = N) -> int:
    if tag == "x_star":
        return int(N) + 1
    return int(tag.split("_", 1)[1])


def ogm_relaxed_constraints_from_observed_pattern(
    lamb_matrix, N: int | sp.Integer
) -> list[str]:
    relaxed_constraints = []
    N = int(N)
    for tag_i in lamb_matrix.row_names:
        i = tag_to_index(tag_i, N)
        if i == N + 1:
            continue
        for tag_j in lamb_matrix.col_names:
            j = tag_to_index(tag_j, N)
            if i < N and i + 1 == j:
                continue
            relaxed_constraints.append(f"f:{tag_i},{tag_j}")
    return relaxed_constraints


pb_prf.set_relaxed_constraints(
    ogm_relaxed_constraints_from_observed_pattern(lamb_dense, N)
)
result = pb_prf.solve(resolve_parameters={"L": L_value, "R": R_value})

# Dual variable associated with the initial condition.
tau_sol = result.dual_var_manager.dual_value("initial_condition")
# Dual variables associated with the interpolation conditions of f.
lamb_sol = result.get_scalar_constraint_dual_value_in_numpy(f)
# Dual matrix associated with the Gram matrix.
S_sol = result.get_gram_dual_matrix()

print("sparse optimum:", result.opt_value)
print("tau:", tau_sol)

sparse optimum: 0.018588926661423158
tau: 0.018588155914589344


In [7]:
lamb_sol.pprint()

<IPython.core.display.Math object>

In [8]:
S_sol.pprint()

<IPython.core.display.Math object>

### The Gram dual matrix has rank 1

In [9]:
np.linalg.matrix_rank(S_sol.matrix)

np.int64(1)

---

## Step 1. Propose a candidate Lyapunov function

### Let $\mathcal I_0=\{(\star,0)\},\,\, \mathcal I_k=\{(j,j+1):j=0,\ldots,k-1\}\cup\{(\star,j):j=0,\ldots,k\}$ for $k=1,\dots,N$ and $\mathcal J_k = \emptyset$ for $k=0,\dots,N$.

In [10]:
V_raw = []
partial_sum = lamb_sol("x_star", "x_0") * f.interp_ineq("x_star", "x_0")
V_raw.append(partial_sum)

for step in range(N_int):
    partial_sum += lamb_sol(f"x_{step}", f"x_{step + 1}") * f.interp_ineq(
        f"x_{step}", f"x_{step + 1}"
    )
    partial_sum += lamb_sol("x_star", f"x_{step + 1}") * f.interp_ineq(
        "x_star", f"x_{step + 1}"
    )
    V_raw.append(partial_sum)

print("V_raw contains:", [f"V_{i}" for i in range(len(V_raw))])

V_raw contains: ['V_0', 'V_1', 'V_2', 'V_3', 'V_4', 'V_5']


## Step 2. Check for admissibility

### Sufficiency is immediate because $\mathcal I_N$ contains the adjacent interpolation constraints and all star interpolation constraints kept by the sparse certificate. We next inspect the ranks of the quadratic parts and the function-value support of each raw partial sum.

In [11]:
pm = pf.ExpressionManager(ctx_prf, resolve_parameters={"L": L_value, "R": R_value})
scalar_labels = [str(s) for s in ctx_prf.basis_scalars()]

for k, V in enumerate(V_raw):
    V_eval = pm.eval_scalar(V)
    rank = np.linalg.matrix_rank(V_eval.inner_prod_coords.astype(float), tol=1e-5)
    func_coords = V_eval.func_coords.astype(float)
    support = [scalar_labels[i] for i, val in enumerate(func_coords) if abs(val) > 1e-7]
    print(f"V_{k}: rank={rank}, nonzero function coordinates={support}")

print("\nFunction-value vector at last iteration:")
pf.pprint_labeled_vector(
    pm.eval_scalar(V_raw[N_int]).func_coords.astype(float),
    scalar_labels,
)

V_0: rank=2, nonzero function coordinates=['f(x_star)', 'f(x_0)']
V_1: rank=3, nonzero function coordinates=['f(x_star)', 'f(x_1)']
V_2: rank=3, nonzero function coordinates=['f(x_star)', 'f(x_2)']
V_3: rank=3, nonzero function coordinates=['f(x_star)', 'f(x_3)']
V_4: rank=3, nonzero function coordinates=['f(x_star)', 'f(x_4)']
V_5: rank=2, nonzero function coordinates=['f(x_star)', 'f(x_5)']

Function-value vector at last iteration:


<IPython.core.display.Math object>

## Step 3. Build a set of special candidate vectors

### For OGM1 we intentionally exclude $z_k$. The candidate set is built from $x_0,\ldots,x_N$, $x_\star$, the gradients $\nabla f(x_0),\ldots,\nabla f(x_N)$, and all pairwise differences among these vectors.

In [12]:
ctx_prf.set_as_current()
x = [ctx_prf[f"x_{i}"] for i in range(N_int + 1)]
x_0 = x[0]
x_star = ctx_prf["x_star"]
g = [f.grad(x_i) for x_i in x]

lyap_basis_candidate = list(ctx_prf.basis_vectors())
lyap_basis_candidate += x[1:]

base_count = len(lyap_basis_candidate)
for i, j in itertools.combinations(range(base_count), 2):
    diff = lyap_basis_candidate[i] - lyap_basis_candidate[j]
    lyap_basis_candidate.append(diff)

print(lyap_basis_candidate)

[x_0, x_star, grad_f(x_0), grad_f(x_1), grad_f(x_2), grad_f(x_3), grad_f(x_4), grad_f(x_5), x_1, x_2, x_3, x_4, x_5, x_0-x_star, x_0-grad_f(x_0), x_0-grad_f(x_1), x_0-grad_f(x_2), x_0-grad_f(x_3), x_0-grad_f(x_4), x_0-grad_f(x_5), x_0-(x_1), x_0-(x_2), x_0-(x_3), x_0-(x_4), x_0-(x_5), x_star-grad_f(x_0), x_star-grad_f(x_1), x_star-grad_f(x_2), x_star-grad_f(x_3), x_star-grad_f(x_4), x_star-grad_f(x_5), x_star-(x_1), x_star-(x_2), x_star-(x_3), x_star-(x_4), x_star-(x_5), grad_f(x_0)-grad_f(x_1), grad_f(x_0)-grad_f(x_2), grad_f(x_0)-grad_f(x_3), grad_f(x_0)-grad_f(x_4), grad_f(x_0)-grad_f(x_5), grad_f(x_0)-(x_1), grad_f(x_0)-(x_2), grad_f(x_0)-(x_3), grad_f(x_0)-(x_4), grad_f(x_0)-(x_5), grad_f(x_1)-grad_f(x_2), grad_f(x_1)-grad_f(x_3), grad_f(x_1)-grad_f(x_4), grad_f(x_1)-grad_f(x_5), grad_f(x_1)-(x_1), grad_f(x_1)-(x_2), grad_f(x_1)-(x_3), grad_f(x_1)-(x_4), grad_f(x_1)-(x_5), grad_f(x_2)-grad_f(x_3), grad_f(x_2)-grad_f(x_4), grad_f(x_2)-grad_f(x_5), grad_f(x_2)-(x_1), grad_f(x_2)-(x_

## Step 4. Find a basis of $\mathbf V_k$ within the candidate vectors

### Without $z_k$, the ordinary candidate search does not reveal a full rank-3 basis. For $k=1,\ldots,N-1$, it reliably finds $\nabla f(x_k)$ and $x_0-x_\star$, so the last direction must be completed from the range of $\mathbf V_k$.

In [13]:
for k, V in enumerate(V_raw):
    aligned_special_vectors = lu.vectors_in_column_space(
        V,
        lyap_basis_candidate,
        ctx_prf,
        resolve_parameters=pm.resolve_parameters,
        rtol=1e-5,
        atol=1e-5,
    )
    print(f"V_{k}:", aligned_special_vectors)

V_0: [grad_f(x_0), x_0-x_star, x_0-(x_1), x_star-(x_1)]
V_1: [grad_f(x_0), grad_f(x_1), x_0-x_star, x_0-(x_1), x_0-(x_2), x_star-(x_1), x_star-(x_2), grad_f(x_0)-grad_f(x_1), x_1-(x_2)]
V_2: [grad_f(x_2), x_0-x_star]
V_3: [grad_f(x_3), x_0-x_star]
V_4: [grad_f(x_4), x_0-x_star]
V_5: [x_0-x_star]


In [14]:
vector_coord_labels = [str(v) for v in ctx_prf.basis_vectors()]
raw_completed_vectors = {}
for k, V in enumerate(V_raw):
    if k == 0 or k == len(V_raw) - 1:
        continue

    fixed_vectors = [g[k], x_0 - x_star]
    v_k, C_k = lu.complete_basis_with_sparsifying_last_vector(
        V,
        fixed_vectors,
        pep_context=ctx_prf,
        resolve_parameters=pm.resolve_parameters,
        rtol=1e-5,
        normalize_last=True,
    )
    raw_completed_vectors[k] = v_k
    labels_k = [str(v) for v in fixed_vectors] + [rf"v_{{{k}}}"]

    pf.pprint_labeled_matrix(C_k, labels_k, labels_k)
    print(rf"v_{{{k}}} coordinates:")
    pf.pprint_labeled_vector(v_k, vector_coord_labels)
    print()

<IPython.core.display.Math object>

v_{1} coordinates:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

v_{2} coordinates:


...

v_{3} coordinates:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

v_{4} coordinates:


<IPython.core.display.Math object>

### The basis completion identifies the extra direction $v_k$, but the raw candidates also contain a constant $\|x_0-x_\star\|^2$ term. We apply translation by this amount and repeat the previous steps

In [15]:
lyap = [V + tau_sol * (x_0 - x_star) ** 2 for V in V_raw]

for k, V in enumerate(lyap):
    V_eval = pm.eval_scalar(V)
    rank = np.linalg.matrix_rank(V_eval.inner_prod_coords.astype(float), tol=1e-5)
    print(f"tilde V_{k}: rank={rank}")

tilde V_0: rank=2
tilde V_1: rank=2
tilde V_2: rank=2
tilde V_3: rank=2
tilde V_4: rank=2
tilde V_5: rank=1


In [16]:
vector_coord_labels = [str(v) for v in ctx_prf.basis_vectors()]


def orient_against_x_star(v_coords):
    x_star_coords = pm.eval_vector(x_star).coords.astype(float).ravel()
    return -v_coords if float(v_coords @ x_star_coords) > 0 else v_coords


completed_vectors = {}
completed_coefficients = {}

for k, V in enumerate(lyap):
    if k == len(lyap) - 1:
        continue

    aligned_special_vectors = lu.vectors_in_column_space(
        V,
        lyap_basis_candidate,
        ctx_prf,
        resolve_parameters=pm.resolve_parameters,
        rtol=1e-5,
        atol=1e-5,
    )
    print(f"tilde V_{k} candidate vectors:", aligned_special_vectors)

    fixed_vectors = [g[k]]
    v_k, C_k = lu.complete_basis_with_sparsifying_last_vector(
        V,
        fixed_vectors,
        pep_context=ctx_prf,
        resolve_parameters=pm.resolve_parameters,
        rtol=1e-5,
        normalize_last=True,
    )
    v_k = orient_against_x_star(v_k)
    completed_vectors[k] = v_k
    completed_coefficients[k] = C_k

    labels_k = [str(v) for v in fixed_vectors] + [rf"v_{k}"]
    pf.pprint_labeled_matrix(C_k, labels_k, labels_k)
    print(rf"v_{k} coordinates:")
    pf.pprint_labeled_vector(v_k, vector_coord_labels)
    print()

tilde V_0 candidate vectors: [grad_f(x_0), x_0-x_star, x_0-(x_1), x_star-(x_1)]


<IPython.core.display.Math object>

v_0 coordinates:


<IPython.core.display.Math object>


tilde V_1 candidate vectors: [grad_f(x_1)]


...

<IPython.core.display.Math object>


tilde V_4 candidate vectors: [grad_f(x_4)]


<IPython.core.display.Math object>

v_4 coordinates:


<IPython.core.display.Math object>

In [17]:
def projection_residual_onto(direction, vector):
    direction = np.asarray(direction, dtype=float).ravel()
    vector = np.asarray(vector, dtype=float).ravel()
    coeff = float(vector @ direction / (direction @ direction))
    residual = np.linalg.norm(vector - coeff * direction)
    return coeff, residual


print("Testing if v_{k+1}-v_k is indeed (numerically) parallel to grad_f(x_{k+1}):")
for k in range(N_int - 1):
    step_vector = completed_vectors[k + 1] - completed_vectors[k]
    grad_direction = pm.eval_vector(g[k + 1]).coords.astype(float).ravel()
    coeff, residual = projection_residual_onto(grad_direction, step_vector)
    print(f"k={k}: coefficient={coeff:.8f}, residual={residual:.2e}")

Testing if v_{k+1}-v_k is indeed (numerically) parallel to grad_f(x_{k+1}):
k=0: coefficient=-0.44120000, residual=3.45e-07
k=1: coefficient=-0.59812350, residual=4.96e-08
k=2: coefficient=-0.74980368, residual=3.45e-08
k=3: coefficient=-0.89843618, residual=2.93e-07


### The numerical basis completion suggests the Lyapunov skeleton $\widetilde V_k=a_k(f(x_k)-f(x_\star))+b_k\|\nabla f(x_k)\|^2+\|v_k\|^2$, with $v_{k+1}=v_k-\eta_k\nabla f(x_{k+1})$ for some $\eta_k > 0$. The analytic step below identifies $v_k$ and verifies the full Lyapunov identity from OGM1 alone.

---

## Step 5. Analytic proof

### As in the case of OGM2, the analytic form of multipliers are
$$
    \lambda_{i,i+1}=\frac{2\theta_i^2}{\theta_N^2}, \qquad
    \lambda_{\star,i}=\lambda_{i,i+1}-\lambda_{i-1,i}=\frac{2\theta_i}{\theta_N^2}\quad(0\le i\le N-1), \qquad
    \lambda_{\star,N}=\frac{1}{\theta_N},
$$
### where $\lambda_{-1,0}=0$.

In [18]:
def lambda_adj_formula(i: int, N: int):
    if 0 <= i <= N - 1:
        return 2 * theta(i, N) ** 2 / theta(N, N) ** 2
    return 0


def lambda_star_formula(i: int, N: int):
    if i == 0:
        return lambda_adj_formula(0, N)
    if 1 <= i <= N - 1:
        return lambda_adj_formula(i, N) - lambda_adj_formula(i - 1, N)
    if i == N:
        return 1 - lambda_adj_formula(N - 1, N)
    return 0


def lambda_formula(tag_i: str, tag_j: str, N: int):
    i = tag_to_index(tag_i, N)
    j = tag_to_index(tag_j, N)
    if tag_i == "x_star" and 0 <= j <= N:
        return lambda_star_formula(j, N)
    if 0 <= i <= N - 1 and j == i + 1:
        return lambda_adj_formula(i, N)
    return 0


lamb_cand = pf.pprint_labeled_matrix(
    lambda tag_i, tag_j: lambda_formula(tag_i, tag_j, N_int),
    lamb_sol.row_names,
    lamb_sol.col_names,
    return_matrix=True,
)
print(
    "closed-form lambdas match numerical certificate?",
    np.allclose(np.asarray(lamb_cand, dtype=float), lamb_sol.matrix, atol=1e-4),
)

tau_formula = L_value / (2 * theta(N_int, N_int) ** 2)
print(
    "closed-form tau matches numerical certificate?", np.allclose(tau_formula, tau_sol)
)
print("tau formula:", tau_formula)
print("tau numerical:", tau_sol)

<IPython.core.display.Math object>

closed-form lambdas match numerical certificate? True
closed-form tau matches numerical certificate? True
tau formula: 0.0185881366636511
tau numerical: 0.018588155914589344


### Symbolic verification helpers

In [19]:
def simplify_sympy_matrix(array):
    return sp.Matrix(array).applyfunc(
        lambda entry: sp.factor(sp.together(sp.nsimplify(entry)))
    )


def reduce_by_quadratic_recurrence(expr, eliminated_symbol, recurrence_expr):
    expr = sp.factor(sp.together(sp.simplify(expr)))
    numerator, denominator = sp.fraction(expr)
    if numerator == 0:
        return sp.S(0)
    try:
        numerator_reduced = sp.rem(
            sp.Poly(sp.expand(numerator), eliminated_symbol),
            sp.Poly(recurrence_expr, eliminated_symbol),
        ).as_expr()
        return sp.factor(sp.together(sp.simplify(numerator_reduced / denominator)))
    except sp.PolynomialError:
        return expr


def symbolic_parts(expr, ctx, resolve_parameters):
    pm_check = pf.ExpressionManager(ctx, resolve_parameters=resolve_parameters)
    matrix_part = simplify_sympy_matrix(pm_check.eval_scalar(expr).inner_prod_coords)
    func_part = simplify_sympy_matrix(pm_check.eval_scalar(expr).func_coords)
    return matrix_part, func_part


def symbolic_vector_coords(vec, ctx, resolve_parameters):
    pm_check = pf.ExpressionManager(ctx, resolve_parameters=resolve_parameters)
    return simplify_sympy_matrix(pm_check.eval_vector(vec).coords)


def display_symbolic_residual(name, matrix_part, func_part):
    print(name)
    print("quadratic residual:")
    display(matrix_part)
    print("function-value residual:")
    display(func_part)
    print(
        "identity verified?",
        matrix_part == sp.zeros(*matrix_part.shape)
        and func_part == sp.zeros(*func_part.shape),
    )


def display_symbolic_vector_residual(name, vector_part):
    print(name)
    display(vector_part)
    print("identity verified?", vector_part == sp.zeros(*vector_part.shape))

### In the paper, we have derived
$$
    v_k=\alpha\Bigl(\theta_{k+1}x_{k+1}-(\theta_{k+1}-1)x_k+\frac{\theta_{k+1}-1}{L}\nabla f(x_k)-x_\star\Bigr)
$$
### and
$$
    v_{k+1}=\alpha\Bigl(\theta_{k+2}x_{k+2}-(\theta_{k+2}-1)x_{k+1}+\frac{\theta_{k+2}-1}{L}\nabla f(x_{k+1})-x_\star\Bigr) .
$$
### We later impose $\alpha^2=L/(2\theta_N^2)$. In particular, substituting the OGM1 update rule gives $v_{k+1}=v_k-2\theta_{k+1}\alpha\nabla f(x_{k+1})/L$ as we verify below.

In [20]:
ctx_ogm1_vector = pf.PEPContext("ogm1_completed_vector_recurrence").set_as_current()

x_k_vec = pf.Vector(is_basis=True, tags=["x_k"])
y_k_vec = pf.Vector(is_basis=True, tags=["y_k"])
x_star_vec = pf.Vector(is_basis=True, tags=["x_{star}"])
g_k_vec = pf.Vector(is_basis=True, tags=["g_k"])
g_kp1_vec = pf.Vector(is_basis=True, tags=["g_{k+1}"])

theta_k_par = pf.Parameter("theta_k")
theta_kp1_par = pf.Parameter("theta_{k+1}")
theta_kp2_par = pf.Parameter("theta_{k+2}")
alpha_par = pf.Parameter("alpha")

y_kp1_vec = x_k_vec - sp.S(1) / L * g_k_vec
x_kp1_vec = (
    y_kp1_vec
    + (theta_k_par - 1) / theta_kp1_par * (y_kp1_vec - y_k_vec)
    + theta_k_par / theta_kp1_par * (y_kp1_vec - x_k_vec)
)
y_kp2_vec = x_kp1_vec - sp.S(1) / L * g_kp1_vec
x_kp2_vec = (
    y_kp2_vec
    + (theta_kp1_par - 1) / theta_kp2_par * (y_kp2_vec - y_kp1_vec)
    + theta_kp1_par / theta_kp2_par * (y_kp2_vec - x_kp1_vec)
)

v_k_vec = alpha_par * (
    theta_kp1_par * x_kp1_vec
    - (theta_kp1_par - 1) * x_k_vec
    + (theta_kp1_par - 1) / L * g_k_vec
    - x_star_vec
)
v_kp1_vec = alpha_par * (
    theta_kp2_par * x_kp2_vec
    - (theta_kp2_par - 1) * x_kp1_vec
    + (theta_kp2_par - 1) / L * g_kp1_vec
    - x_star_vec
)

completed_vector_residual = (
    v_kp1_vec - v_k_vec + 2 * theta_kp1_par * alpha_par / L * g_kp1_vec
)

In [21]:
L_sym = sp.Symbol("L")
theta_k_sym = sp.Symbol(r"\theta_k")
theta_kp1_sym = sp.Symbol(r"\theta_{k+1}")
theta_kp2_sym = sp.Symbol(r"\theta_{k+2}")
theta_N_sym = sp.Symbol(r"\theta_N")
alpha_sym = sp.Symbol(r"\alpha")

vector_resolve = {
    "L": L_sym,
    "theta_k": theta_k_sym,
    "theta_{k+1}": theta_kp1_sym,
    "theta_{k+2}": theta_kp2_sym,
    "alpha": alpha_sym,
}
completed_vector_coords = symbolic_vector_coords(
    completed_vector_residual,
    ctx_ogm1_vector,
    vector_resolve,
)
display_symbolic_vector_residual(
    "completed-vector recurrence residual",
    completed_vector_coords,
)

completed-vector recurrence residual


Matrix([
[0],
[0],
[0],
[0],
[0]])

identity verified? True


### Find the Lyapunov coefficients for $0\le k\le N-2$ by solving
$$
0=\widetilde V_k-\widetilde V_{k+1}-\lambda_{k,k+1}I_{k,k+1}-\lambda_{\star,k+1}I_{\star,k+1}.
$$
### Here $I_{k,k+1}$ and $I_{\star,k+1}$ are the nonnegative interpolation inequalities. The square term is written with an unknown coefficient $c$ first, and the system itself recovers $c=L/(2\theta_N^2)$.

In [22]:
ctx_ogm1_interior = pf.PEPContext("ogm1_without_z_interior_proof").set_as_current()

x_k_abs = pf.Vector(is_basis=True, tags=["x_k"])
y_k_abs = pf.Vector(is_basis=True, tags=["y_k"])
x_star_abs = pf.Vector(is_basis=True, tags=["x_{star}"])
g_k_abs = pf.Vector(is_basis=True, tags=["g_k"])
g_kp1_abs = pf.Vector(is_basis=True, tags=["g_{k+1}"])

f_k_abs = pf.Scalar(is_basis=True, tags=["f_k"])
f_kp1_abs = pf.Scalar(is_basis=True, tags=["f_{k+1}"])
f_star_abs = pf.Scalar(is_basis=True, tags=["f_{star}"])

theta_k_par = pf.Parameter("theta_k")
theta_kp1_par = pf.Parameter("theta_{k+1}")
theta_kp2_par = pf.Parameter("theta_{k+2}")
lambda_cons_par = pf.Parameter("lambda_{k,k+1}")
lambda_star_par = pf.Parameter("lambda_{star,k+1}")

y_kp1_abs = x_k_abs - sp.S(1) / L * g_k_abs
x_kp1_abs = (
    y_kp1_abs
    + (theta_k_par - 1) / theta_kp1_par * (y_kp1_abs - y_k_abs)
    + theta_k_par / theta_kp1_par * (y_kp1_abs - x_k_abs)
)
y_kp2_abs = x_kp1_abs - sp.S(1) / L * g_kp1_abs
x_kp2_abs = (
    y_kp2_abs
    + (theta_kp1_par - 1) / theta_kp2_par * (y_kp2_abs - y_kp1_abs)
    + theta_kp1_par / theta_kp2_par * (y_kp2_abs - x_kp1_abs)
)

q_k_abs = (
    theta_kp1_par * x_kp1_abs
    - (theta_kp1_par - 1) * x_k_abs
    + (theta_kp1_par - 1) / L * g_k_abs
    - x_star_abs
)
q_kp1_abs = (
    theta_kp2_par * x_kp2_abs
    - (theta_kp2_par - 1) * x_kp1_abs
    + (theta_kp2_par - 1) / L * g_kp1_abs
    - x_star_abs
)

a_k_par = pf.Parameter("a_k")
a_kp1_par = pf.Parameter("a_{k+1}")
b_k_par = pf.Parameter("b_k")
b_kp1_par = pf.Parameter("b_{k+1}")
c_par = pf.Parameter("c")

V_k_abs = a_k_par * (f_k_abs - f_star_abs) + b_k_par * g_k_abs**2 + c_par * q_k_abs**2
V_kp1_abs = (
    a_kp1_par * (f_kp1_abs - f_star_abs)
    + b_kp1_par * g_kp1_abs**2
    + c_par * q_kp1_abs**2
)

I_cons_abs = (
    (f_k_abs - f_kp1_abs)
    + g_kp1_abs * (x_kp1_abs - x_k_abs)
    - sp.S(1) / (2 * L) * (g_k_abs - g_kp1_abs) ** 2
)
I_star_abs = (
    (f_star_abs - f_kp1_abs)
    + g_kp1_abs * (x_kp1_abs - x_star_abs)
    - sp.S(1) / (2 * L) * g_kp1_abs**2
)

interior_diff = (
    V_k_abs - V_kp1_abs - lambda_cons_par * I_cons_abs - lambda_star_par * I_star_abs
)

In [23]:
lambda_cons_sym = sp.Symbol(r"\lambda_{k,k+1}")
lambda_star_sym = sp.Symbol(r"\lambda_{star,k+1}")

a_k_sym, a_kp1_sym, b_k_sym, b_kp1_sym, c_sym = sp.symbols(r"a_k a_{k+1} b_k b_{k+1} c")

interior_resolve = {
    "L": L_sym,
    "theta_k": theta_k_sym,
    "theta_{k+1}": theta_kp1_sym,
    "theta_{k+2}": theta_kp2_sym,
    "lambda_{k,k+1}": lambda_cons_sym,
    "lambda_{star,k+1}": lambda_star_sym,
    "a_k": a_k_sym,
    "a_{k+1}": a_kp1_sym,
    "b_k": b_k_sym,
    "b_{k+1}": b_kp1_sym,
    "c": c_sym,
}

interior_matrix, interior_func = symbolic_parts(
    interior_diff, ctx_ogm1_interior, interior_resolve
)
row_index = [str(v) for v in ctx_ogm1_interior.basis_vectors()]
func_index = [str(s) for s in ctx_ogm1_interior.basis_scalars()]

pf.pprint_labeled_matrix(
    np.array(interior_matrix.tolist(), dtype=object),
    row_index,
    row_index,
    precision=None,
)
pf.pprint_labeled_vector(
    np.array(interior_func, dtype=object).reshape(-1),
    func_index,
    precision=None,
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [24]:
interior_lambda_subs = {
    lambda_cons_sym: 2 * theta_k_sym**2 / theta_N_sym**2,
    lambda_star_sym: 2 * theta_kp1_sym / theta_N_sym**2,
}
interior_recurrence = theta_k_sym**2 - theta_kp1_sym**2 + theta_kp1_sym


def reduce_interior(expr):
    return reduce_by_quadratic_recurrence(
        expr.subs(interior_lambda_subs),
        theta_k_sym,
        interior_recurrence,
    )


coefficient_unknowns = (
    a_k_sym,
    a_kp1_sym,
    b_k_sym,
    b_kp1_sym,
    c_sym,
)
interior_equations = [
    reduce_interior(entry)
    for entry in list(interior_matrix) + list(interior_func)
    if any(entry.has(unknown) for unknown in coefficient_unknowns)
]
interior_equations = [eq for eq in interior_equations if eq != 0]

coefficient_matrix, rhs = sp.linear_eq_to_matrix(
    interior_equations, coefficient_unknowns
)
solution_tuple = next(
    iter(sp.linsolve((coefficient_matrix, rhs), coefficient_unknowns))
)
solved_solution = {
    unknown: sp.factor(sp.together(sp.simplify(expression)))
    for unknown, expression in zip(coefficient_unknowns, solution_tuple)
}

print("coefficient matrix rank:", coefficient_matrix.rank())
print("coefficient nullity:", len(coefficient_matrix.nullspace()))
print("Solved coefficient tuple:")
display(
    ipd.Math(
        r"\begin{aligned}"
        + r"\\".join(
            sp.latex(sp.Eq(unknown, solved_solution[unknown]))
            for unknown in coefficient_unknowns
        )
        + r"\end{aligned}"
    )
)

coefficient matrix rank: 5
coefficient nullity: 0
Solved coefficient tuple:


<IPython.core.display.Math object>

In [25]:
closed_form_solution = {
    a_k_sym: 2 * theta_k_sym**2 / theta_N_sym**2,
    a_kp1_sym: 2 * theta_kp1_sym**2 / theta_N_sym**2,
    b_k_sym: -(theta_k_sym**2) / (L_sym * theta_N_sym**2),
    b_kp1_sym: -(theta_kp1_sym**2) / (L_sym * theta_N_sym**2),
    c_sym: L_sym / (2 * theta_N_sym**2),
}

print("Closed-form coefficients:")
display(
    ipd.Math(
        r"\begin{aligned}"
        + r"\\".join(
            sp.latex(sp.Eq(unknown, closed_form_solution[unknown]))
            for unknown in coefficient_unknowns
        )
        + r"\end{aligned}"
    )
)

print(
    "Solved tuple equals the closed form under the recurrence?",
    all(
        reduce_by_quadratic_recurrence(
            solved_solution[unknown] - closed_form_solution[unknown],
            theta_k_sym,
            interior_recurrence,
        )
        == 0
        for unknown in coefficient_unknowns
    ),
)

Closed-form coefficients:


<IPython.core.display.Math object>

Solved tuple equals the closed form under the recurrence? True


In [26]:
verification_subs = {**interior_lambda_subs, **closed_form_solution}


def verify_interior_entry(entry):
    return reduce_by_quadratic_recurrence(
        entry.subs(verification_subs),
        theta_k_sym,
        interior_recurrence,
    )


interior_matrix_verified = interior_matrix.applyfunc(verify_interior_entry)
interior_func_verified = interior_func.applyfunc(verify_interior_entry)

display_symbolic_residual(
    "OGM1 nonterminal transition residual",
    interior_matrix_verified,
    interior_func_verified,
)

OGM1 nonterminal transition residual
quadratic residual:


Matrix([
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0]])

function-value residual:


Matrix([
[0],
[0],
[0]])

identity verified? True


### Therefore the OGM1 update rule admits the Lyapunov function
$$
    \widetilde V_k = \frac{2\theta_k^2}{\theta_N^2} (f(x_k) - f(x_\star)) - \frac{\theta_k^2}{L\theta_N^2} \|\nabla f(x_k)\|^2
    + \frac{L}{2\theta_N^2} \left\|
        \theta_{k+1}x_{k+1}
        -(\theta_{k+1}-1)x_k
        + \frac{\theta_{k+1}-1}{L}\nabla f(x_k)
        -x_\star
    \right\|^2 .
$$
### The completed vector is exactly the hidden auxiliary term $z_{k+1}-x_\star$ from OGM2, recovered here without placing any $z_k$ vectors in the candidate set.